In [1]:
from profis.utils.modelinit import initialize_model
import pandas as pd
from profis.utils.finger import encode
import torch

In [2]:
krfp = True
device = torch.device('cuda')
if krfp:
    CONFIG_PATH = 'models/best_KRFP_SMILES/hyperparameters.ini'
    TEST_DATA_PATH = 'data/RNN_dataset_KRFP_val_10.parquet'
else:
    CONFIG_PATH = 'models/best_ECFP_SMILES/hyperparameters.ini'
    TEST_DATA_PATH = 'data/RNN_dataset_ECFP_val_10.parquet'

VAE = initialize_model(CONFIG_PATH, device)
if krfp:
    VAE.load_state_dict(torch.load('models/best_KRFP_SMILES/epoch_300.pt'))
else:
    VAE.load_state_dict(torch.load('models/best_ECFP_SMILES/epoch_300.pt'))

In [3]:
df = pd.read_parquet(TEST_DATA_PATH)

In [11]:
mus, _ = encode(df, VAE, device)

In [6]:
encoded_df = pd.DataFrame(mus)

In [7]:
encoded_df

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.768103,-0.033737,-1.434149,0.917166,-0.010640,3.016909,-0.914438,1.713808,-1.064915,0.790062,...,-0.879908,0.332654,0.391890,0.052124,-0.234037,-0.774279,0.005846,-0.047898,0.324873,-0.639922
1,-1.181600,0.030547,0.179476,0.237315,-0.016623,0.444584,0.958251,0.954771,1.820895,-0.853563,...,-0.534760,-1.374929,-0.374127,0.000726,-0.441525,-0.335337,-0.079499,0.073561,0.251041,-0.673712
2,0.931317,-0.049527,1.403122,1.019054,-1.547402,0.671861,0.764202,2.508106,0.402110,-1.541266,...,-0.485299,1.978824,0.874768,0.050256,0.781113,-0.514409,0.060392,0.012787,0.137049,-0.793549
3,-0.407314,-0.065510,0.029250,-1.314012,-2.383688,0.000725,-0.442581,-1.843803,-0.244193,-1.306896,...,0.797325,-0.194531,-0.767411,0.084382,0.150744,-0.722773,0.135505,-0.062390,0.945995,0.241625
4,0.946529,0.018676,-0.594700,-1.136944,-0.742532,0.887144,-1.169989,0.272914,0.708709,0.621666,...,0.084224,2.143038,-0.724375,-0.013156,-0.550997,1.157381,-0.018700,0.060040,0.202112,0.287338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112604,0.810861,-0.022308,-0.427842,0.303673,0.267311,1.636398,-0.974161,-0.028127,0.740578,-0.069361,...,0.095644,1.247475,1.092235,-0.001239,-0.413853,1.026526,-0.003745,0.022593,-0.233319,2.061632
112605,0.506146,-0.016903,-1.199212,-0.693626,0.112404,-0.068821,0.758645,0.860616,1.433452,0.560324,...,1.637784,1.443076,-0.704412,0.012272,-0.313563,1.178038,0.012416,0.089579,0.870621,0.578214
112606,0.849316,0.047752,0.581645,0.138106,-0.466554,-0.491006,-0.453835,-1.001356,1.863222,-1.108004,...,0.565366,1.305645,0.038674,0.006428,0.840286,-0.843655,-0.002173,-0.002099,1.119092,-0.697798
112607,-0.081395,0.008591,-0.685543,0.236492,0.370906,-0.160738,-0.557650,0.792318,-0.509375,-0.364844,...,-0.135503,2.034239,-0.006860,-0.011354,-0.560041,1.434811,-0.018329,0.016035,-2.087095,-1.976028


In [ ]:
from profis.utils.vectorizer import SMILESVectorizer
from profis.gen.dataset import SMILESDataset
from torch.utils.data import DataLoader

vectorizer = SMILESVectorizer(pad_to_len=128)

dataset = SMILESDataset(df, vectorizer)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=False)

reconstructions = []
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        print(f'Batch {i+1}/{len(dataloader)}')
        X, y = batch
        X = X.to(device)
        y_hat, _ = VAE(X, y)
        reconstructions.append(y_hat.detach().cpu().numpy())
len(reconstructions)

Batch 1/110
Batch 2/110
Batch 3/110
Batch 4/110
Batch 5/110
Batch 6/110
Batch 7/110


In [ ]:
from profis.pred.pred import stochastic_decoder
import numpy as np
n_trials = 1000

reconstructions = np.concatenate(reconstructions)
for vector in reconstructions:
    print(stochastic_decoder(vector, vectorizer, n_trials=n_trials))
    

df["smiles"] = [
                stochastic_decoder(vector, vectorizer, n_trials=n_trials)
                for vector in reconstructions
            ]